# Project Part 3: Base Map & Flythrough

## Overview

In Part 2 you produced a cleaned, sentiment-annotated dataset and a written analysis. Now you will build the **base map** and plan the **flythrough** — the scrollytelling experience that becomes the centerpiece of your team website.

This notebook covers:

- Building an interactive bubble map of geoparsed locations, sized by post count and colored by sentiment
- Designing your narrative flythrough: choosing stops, writing chapter descriptions, and exporting the config
- Saving both the base map and the flythrough config so they are ready to publish

---

## ⚠️ Before You Begin

You must have completed **[project_part_1_data_pipeline.ipynb](project_part_1_data_pipeline.ipynb)** and created a cleaned data set of your school's data. You should see a copy of it in your school's data folder called `{SCHOOL}_geoparsed_cleaned_sentiment.pickle`, where `{SCHOOL}` is your school's abbreviation. If along the way you find that the data was not properly cleaned, you have to repeat steps 2–5 in **[project_part_1_data_pipeline.ipynb](project_part_1_data_pipeline.ipynb)**.

In [ ]:
# ============================================================
# STEP 0: Set your school (must match Part 2)
# ============================================================

SCHOOL = "UNC"   # <-- change this to your school

import pandas as pd
import plotly.express as px
import json

---

## 📖 1 Follow Along — Load and Aggregate Your Processed Data

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

The cell below loads sentiment data for **both your school and JMU**, then aggregates each dataset to one row per unique place. The two datasets are combined into a single DataFrame before any classification. This mirrors the approach from [Part 2](project_part_2_whitepaper.ipynb) and ensures that the Jenks size and color bins computed in Section 2 are shared across both schools — so equivalent bubbles carry equivalent meaning on the map.

The `school` column records which school each place belongs to; it will appear in the hover tooltip.

If your school's file is not found, complete `project_part_1_data_pipeline.ipynb` for your school first.

In [ ]:
import os
import numpy as np

SCHOOL_DATA_PATH = f'../data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned_sentiment.pickle'
JMU_PICKLE_PATH = "../data/JMU/JMU_geoparsed_cleaned_sentiment.pickle"
JMU_CSV_PATH = "../data/JMU/JMU_geoparsed_cleaned_sentiment.csv"
JMU_BACKUP_PATH = "../data/JMU/JMU_geoparsed_cleaned_sentiment.csv"

df_places = None

def _aggregate(df_raw, school_name):
    """Aggregate a per-sentence DataFrame to one row per place."""
    return (
        df_raw
        .dropna(subset=["place", "latitude", "longitude"])
        .astype({"latitude": float, "longitude": float})
        .groupby("place", sort=False)
        .agg(
            location_count=("place", "size"),
            latitude=("latitude", "first"),
            longitude=("longitude", "first"),
            sentences=("sentences", lambda x: " | ".join(str(s) for s in list(x)[:5])),
            avg_roberta_compound=("roberta_compound", "mean"),
            place_type=("place_type", "first"),
        )
        .reset_index()
        .assign(school=school_name)
    )

# ── Load school data ──────────────────────────────────────────────────────────
if not os.path.exists(SCHOOL_DATA_PATH):
    print(f'⛔ File not found: {SCHOOL_DATA_PATH}')
    print("   Complete project_part_1_data_pipeline.ipynb for your school first.")
else:
    df_school_raw = pd.read_pickle(SCHOOL_DATA_PATH)
    print(f'✅ {SCHOOL}: loaded {len(df_school_raw):,} rows')

    # ── Load JMU data (pickle → primary CSV → backup CSV) ────────────────────────
    if os.path.exists(JMU_PICKLE_PATH):
        df_jmu_raw = pd.read_pickle(JMU_PICKLE_PATH)
    elif os.path.exists(JMU_CSV_PATH):
        df_jmu_raw = pd.read_csv(JMU_CSV_PATH)
        print("⚠️  Using JMU primary CSV (pickle not found).")
    elif os.path.exists(JMU_BACKUP_PATH):
        df_jmu_raw = pd.read_csv(JMU_BACKUP_PATH)
        print("⚠️  Using JMU backup sentiment data.")
    else:
        raise FileNotFoundError("No JMU sentiment data found in ../data/JMU/.")
    print(f'✅ JMU:  loaded {len(df_jmu_raw):,} rows')

    # ── Aggregate each school to one row per place ────────────────────────────────
    df_school = _aggregate(df_school_raw, SCHOOL)
    df_jmu    = _aggregate(df_jmu_raw,    "JMU")

    # ── Combine — Jenks bins in Section 2 will be shared across both schools ──────
    df_places = pd.concat([df_jmu, df_school], ignore_index=True)

    df_places = df_places.reset_index(drop=True)
    df_places["id"] = df_places["school"] + (df_places.groupby("school").cumcount() + 1).astype(str)
    df_places.to_csv (f'../data/{SCHOOL}/unique_place_ids_JMU_{SCHOOL}.csv')
    print(f'\n✅ JMU: {len(df_jmu):,} unique places  |  {SCHOOL}: {len(df_school):,} unique places')
    print(
        f"List of unique place ideas created in: ../data/{SCHOOL}/unique_place_ids_JMU_{SCHOOL}.csv"
    )

### ✍️ 2.1 Critical Activity - Completing the Design Brief

Fill in the design brief table below **before** changing any code. Every parameter should be a deliberate choice — not an accepted default. Reference your observations from [Lesson 6](../lesson_6_mapping_fundamentals/lesson_6_mapping_fundamentals.ipynb) Decisions 1–7.

Complete every row in the design brief, then set each matching variable in the code cell and run it to generate your map.

### Design Brief

**Fill in every row before touching the code cell.** Reference Decisions 1–7 from Lesson 6.

| Design Decision | Variable | Your Choice | Reasoning |
|---|---|---|---|
| Filtering threshold | `MIN_COUNT` | | Which places are worth showing? |
| Count ceiling | `MAX_COUNT` | | Cap outlier locations that would dominate the size scale — leave blank for no limit |
| Place type filter | `PLACE_TYPES` | | Which geographic scales belong in your story? |
| Size classification | `N_SIZE_CLASSES` | | How many Jenks size classes? (3–5) |
| Color buckets | `N_COLOR_BUCKETS` | | How many sentiment color classes? (3–7) |
| Color scale | `COLOR_SCALE` | | Which scale is honest and accessible? |
| Maximum bubble size | `SIZE_MAX` | | How large should the biggest bubble be in pixels? |
| Minimum bubble size | `SIZE_MIN` | | Smallest visible bubble — keeps low-count locations readable |
| Base map style | `MAP_STYLE` | | What tone does the background set? |
| Center coordinates | `CENTER` | | Reference viewport — flythrough will override |
| Zoom level | `ZOOM` | | Reference zoom — flythrough will override |

> 👉 **Note:** *Your submitted map must differ from the default values in at least three deliberate ways, each justifiable from this brief.*


In [ ]:

import mapclassify
import plotly.colors as pc

if df_places is None:
    print("⛔ No data — run the cells in Section 1 first.")
else:
    # ── Your design decisions — fill in every value, then run ────────────────────
    MIN_COUNT       = 3              # ← Decision 1: minimum post count per location
    MAX_COUNT       = None           # ← count ceiling — None keeps all; try 200–500 to suppress outliers
    PLACE_TYPES     = None           # ← Decision 2: list e.g. ['City', 'Building'], or None for all types
    N_SIZE_CLASSES  = 4              # ← Decision 3: number of Jenks size classes (try 3–5)
    N_COLOR_BUCKETS = 5              # ← Decision 4: number of sentiment color buckets (try 3–7)
    COLOR_SCALE     = "RdYlGn"      # ← Decision 5: 'RdYlGn', 'RdBu', 'Spectral', 'Viridis'
    SIZE_MAX        = 18             # ← maximum bubble diameter in pixels (try 12–30)
    SIZE_MIN        = 6              # ← minimum bubble diameter in pixels (try 2–10)
    MAP_STYLE       = "carto-positron"  # ← Decision 6: 'carto-positron', 'carto-darkmatter', 'open-street-map'
    CENTER          = {  # ← reference only; flythrough will override this
        "lat": 37.5,
        "lon": -78.0,
    }
    ZOOM            = 6              # ← reference only; flythrough will override this

    # ── Filter ────────────────────────────────────────────────────────────────────
    _count_mask = df_places["location_count"] >= MIN_COUNT
    if MAX_COUNT is not None:
        _count_mask = _count_mask & (df_places["location_count"] <= MAX_COUNT)
    if PLACE_TYPES is not None:
        df_work = df_places[_count_mask & df_places["place_type"].isin(PLACE_TYPES)].copy()
    else:
        df_work = df_places[_count_mask].copy()

   
    # ── Size classification — Jenks on combined data ──────────────────────────────
    _jnb_s = mapclassify.NaturalBreaks(df_work["location_count"].values, k=N_SIZE_CLASSES)
    df_work["size_class"] = (_jnb_s.yb + 1).astype(float)

    # ── Color classification — Jenks on combined data ─────────────────────────────
    scores  = df_work["avg_roberta_compound"]
    _jnb_c  = mapclassify.NaturalBreaks(scores.values, k=N_COLOR_BUCKETS)
    _breaks = _jnb_c.bins
    _lo     = scores.min()
    _labels = []
    for _hi in _breaks:
        _labels.append(f"{_lo:.2f} to {_hi:.2f}")
        _lo = _hi
    df_work["color_class"] = pd.cut(scores, bins=[-float("inf")] + list(_breaks), labels=_labels)
    _palette   = pc.sample_colorscale(COLOR_SCALE, [i / (N_COLOR_BUCKETS - 1) for i in range(N_COLOR_BUCKETS)])
    _color_map = {label: color for label, color in zip(_labels, _palette)}

    # ── Build the map ─────────────────────────────────────────────────────────────
    fig = px.scatter_map(
        df_work,
        lat="latitude", lon="longitude",
        size="size_class",
        color="color_class",
        hover_name="place",
        hover_data={
            "id": True,
            "school": True,
            "avg_roberta_compound": ":.3f",
            "location_count": True,
            "place_type": True,
            "size_class": False,
            "color_class": False,
            "latitude": False,
            "longitude": False,
        },
        color_discrete_map=_color_map,
        category_orders={
            "color_class": _labels,
        },
        size_max=SIZE_MAX,
        map_style=MAP_STYLE,
        center=CENTER,
        zoom=ZOOM,
        height=650,
        title=f'JMU & {SCHOOL} — Base Map  ({len(df_work):,} locations)',
    )
    fig.update_traces(marker_sizemin=SIZE_MIN)
    fig.update_layout(
        margin={
            "r": 0,
            "t": 50,
            "l": 0,
            "b": 0,
        }
    )
    fig.show()


---

## ✍️ 3 Plan Your Flythrough Journey

The flythrough is your team's guided tour through the data. Rather than automatically selecting locations by score or count, you choose the stops because you have a story to tell.

### Step 1 — Explore the map above

Hover over locations that interest you. The hover tooltip shows each location's **ID** — a number you will use below to reference it. Note down:

- The ID
- The place name (exactly as it appears in the hover)
- Why you find it interesting — sentiment, count, location, connection to your hypothesis

Aim for **5–10 stops** that form a coherent narrative. Think spatially: a good flythrough has a sense of movement (e.g. local → regional → national, or JMU → your school → comparison).

### Step 2 — Gather your images

For each stop you plan to include, you need **one image**. Upload your images to the `project_mapping_emotions/images/` folder in your repository. Name them clearly (e.g. `dhall.jpg`, `franklin_street.jpg`). You can use photos you took, Creative Commons images, or screenshots from Google Maps Street View.

### Step 3 — Fill in `FLYTHROUGH_LOCATIONS` below

Each entry in the list is one stop on your tour. Fill in all four fields:

| Field | What to enter |
|---|---|
| `id` | The number from the hover tooltip |
| `title` | The title displayed on the chapter card |
| `description` | 2–3 sentences describing the place and what the data reveals about it |
| `media` | Relative path to your image, e.g. `'images/dhall.jpg'` |

> 👉 **Note:** *The code cell below validates your entries and generates the JSON for `interactive_tour_config.js`. If an ID is not found, it will tell you which one failed.*


In [ ]:
# ── Quick reference: all available fields ────────────────────────────────────
#
#   DATA & CAMERA
#   'show_data':        'all_locations'          'all_locations' (default), 'jmu_locations',
#                                                'non_jmu_locations'
#   'date_range':       (2020, 2022)             filter data to this year range only
#   'zoom':             15                       camera zoom level (default: 15)
#   'center':           {'latitude': 0.0,        override the camera center point
#                        'longitude': 0.0}
#
#   MAP MARKERS
#   'opacity':          0.3                      dim background markers (0.0–1.0); any value
#                                                < 1.0 also enables the focal highlight marker
#   'hide_place_types': ['Building']             hide markers of this geographic type
#   'label':            True                     pin the place name to the map
#   'pulse':            True                     animate the highlighted marker
#
#   CHAPTER CARD
#   'title':            'Chapter Title'          title displayed on the chapter card
#   'media':            'images/file.jpg'        image displayed on the card
#   'caption':          'Photo: Author, CC BY'   credit line shown below the image
#   'quote':            'A quote from the data'  pull quote displayed on the card
#   'show_stats':       True                     post count and sentiment badge
#   'show_place_type':  True                     geographic type badge (Building, City, etc.)
#
#   TRANSITION
#   'transition':       'normal'                 'slow', 'normal' (default), or 'fast'
#
FLYTHROUGH_LOCATIONS = [
    # ── Chapter Card Options ──────────────────────────────────────────────────────
    {
        "chapter": "Introduction",
        "id": "JMU1",
        "title": "Introduction",
        "description": "Your interactive tour should be a self-contained story about your data. The story is driven by two main components: the scrolling text boxes on the left (chapters) and the map on the right. Both can be customized to fit your particular story and this interactive tour will cycle through some of the options. You should consider how you visualize your story very carefully. You want to provide enough information to your users without overwhelming them.",
        "media": "images/quad.jpg",
        "zoom": 2,
        "center": {"latitude": 37.5, "longitude": -78.0},
        "show_data": "all_locations",
    },
    {
        "chapter": "Basic Setup",
        "id": "JMU26",
        "title": "Basic Setup",
        "description": 'Each chapter entry has four required fields:<br><pre><code>{\n    "chapter":     "Basic Setup",   # name used for internal purposes, not visible to user\n    "id":          "JMU26",          # centers the map to that location at zoom 15\n    "title":       "Chapter Title",  # title displayed on the chapter card\n    "description": "Your text ...",  # body text of the chapter card\n}</code></pre>',
    },
    {
        "chapter": "Image Setup",
        "id": "JMU26",
        "title": "Image Setup",
        "description": 'Each chapter can display one image, optimized for a 16:9 aspect ratio. Save your image to the <code>images/</code> folder and reference it with the <code>media</code> field. Use the optional <code>caption</code> field to add a credit line below the image:<br><pre><code>"media":   "images/bridgeforth-stadium.jpg",\n"caption": "Photo: Author Name, CC BY 2.0"</code></pre>',
        "media": "images/bridgeforth-stadium.jpg",
        "caption": "Bridgeforth Stadium",
    },
    {
        "chapter": "Other Options",
        "id": "JMU26",
        "title": "Other Chapter Options",
        "description": 'Three optional fields add extra context to each chapter card. Use <code>quote</code> to surface a representative sentence from the data as a styled pull quote. Add <code>show_stats</code> to display a post count and sentiment badge, and <code>show_place_type</code> to show a geographic type badge:<br><pre><code>"quote":           "A representative quote from the data.",\n"show_stats":      True,  # post count and sentiment badge\n"show_place_type": True   # geographic type badge (Building, City, etc.)</code></pre>',
        "media": "images/bridgeforth-stadium.jpg",
        "caption": "Bridgeforth Stadium",
        "quote": "A representative quote from the data that stands out.",
        "show_stats": True,
        "show_place_type": True,
    },
    # ── Movement Options ──────────────────────────────────────────────────────────
    {
        "chapter": "Camera: Location ID",
        "id": "JMU29",
        "title": "Moving the Camera",
        "description": 'The camera moves to the location identified by <code>id</code>. Change the id to any location visible on the map — hover over a marker to see its id. Use <code>transition</code> to control the animation speed:<br><pre><code>"id":         "JMU29",\n"transition": "slow"   # "slow", "normal" (default), or "fast"</code></pre>',
    },
    {
        "chapter": "Camera: Custom Center",
        "id": "JMU1",
        "title": "Custom Center & Zoom",
        "description": 'By default the camera centers on the <code>id</code> location. Use <code>center</code> and <code>zoom</code> to override this — useful when your story requires a regional view with no single pin. Here the camera is set to the Virginia–North Carolina border to frame both states simultaneously:<br><pre><code>"center": {"latitude": 36.54, "longitude": -79.0},\n"zoom":   7</code></pre>',
        "zoom": 7,
        "center": {"latitude": 36.54, "longitude": -79.0},
        "show_data": "all_locations",
    },
    # ── Map Representation ────────────────────────────────────────────────────────
    {
        "chapter": "All Locations",
        "id": "JMU1",
        "title": "All Locations",
        "description": '<code>show_data</code> controls which markers are visible at each stop. Setting it to <code>"all_locations"</code> renders every marker from both schools simultaneously — pair it with a wide zoom and custom center to frame the full dataset:<br><pre><code>"show_data": "all_locations",\n"zoom":      7,\n"center":    {"latitude": 37.5, "longitude": -78.0}</code></pre>',
        "zoom": 7,
        "center": {"latitude": 37.5, "longitude": -78.0},
        "show_data": "all_locations",
    },
    {
        "chapter": "JMU Locations",
        "id": "JMU1",
        "title": "JMU Locations",
        "description": 'Setting <code>show_data</code> to <code>"jmu_locations"</code> hides your school\'s markers and shows only JMU data. Keeping the same <code>center</code> and <code>zoom</code> as the previous stop lets readers directly compare the two geographic distributions:<br><pre><code>"show_data": "jmu_locations"</code></pre>',
        "zoom": 7,
        "center": {"latitude": 37.5, "longitude": -78.0},
        "show_data": "jmu_locations",
    },
    {
        "chapter": "UNC Locations",  # ← change UNC to your school abbreviation
        "id": "JMU1",
        "title": "UNC Locations",  # ← change UNC to your school abbreviation
        "description": 'Setting <code>show_data</code> to <code>"non_jmu_locations"</code> shows only your school\'s markers. Cycling through all three stops creates a side-by-side regional comparison without the camera ever moving:<br><pre><code>"show_data": "non_jmu_locations"</code></pre>',
        "zoom": 7,
        "center": {"latitude": 37.5, "longitude": -78.0},
        "show_data": "non_jmu_locations",
    },
    {
        "chapter": "Cities & States Only",
        "id": "JMU1",
        "title": "Filtering by Place Type",
        "description": '<code>hide_place_types</code> removes markers of the listed geographic types from the map. Here Buildings, Roads, and Neighborhoods are hidden to surface only City- and State-level patterns and remove map clutter:<br><pre><code>"hide_place_types": ["Building", "Road", "Neighborhood"]</code></pre>',
        "zoom": 7,
        "center": {"latitude": 37.5, "longitude": -78.0},
        "show_data": "all_locations",
        "hide_place_types": ["Building", "Road", "Neighborhood"],
    },
    # ── Marker Options ────────────────────────────────────────────────────────────
    {
        "chapter": "Cluster: Opacity",
        "id": "JMU26",
        "title": "Marker Opacity",
        "description": '<code>opacity</code> dims all background markers so the highlighted location stands out. A value of <code>0.5</code> gives enough surrounding context without competing with the focus location — use <code>0.0</code> to hide all others entirely:<br><pre><code>"opacity":   0.5,\n"show_data": "all_locations"</code></pre>',
        "media": "images/bridgeforth-stadium.jpg",
        "opacity": 0.5,
        "show_data": "all_locations",
    },
    {
        "chapter": "Cluster: Opacity + Label",
        "id": "JMU26",
        "title": "Opacity & Label",
        "description": '<code>label</code> pins the place name directly onto the map. Combine it with <code>opacity</code> so readers can identify the location without needing to hover over the marker:<br><pre><code>"opacity": 0.5,\n"label":   True</code></pre>',
        "media": "images/bridgeforth-stadium.jpg",
        "opacity": 0.5,
        "label": True,
        "show_data": "all_locations",
    },
    {
        "chapter": "Cluster: Full",
        "id": "JMU26",
        "title": "Opacity, Label & Pulse",
        "description": '<code>pulse</code> adds a radiating ring animation to the highlighted marker, immediately drawing the eye to the location. Use it sparingly for stops where the exact geographic position is the main point of the chapter:<br><pre><code>"opacity": 0.5,\n"label":   True,\n"pulse":   True</code></pre>',
        "media": "images/d-hall.jpg",
        "opacity": 0.5,
        "label": True,
        "pulse": True,
        "show_data": "all_locations",
    },
    # ── Temporal Slicing ──────────────────────────────────────────────────────────
    # All three stops point to the same location; only the date window changes.
    # The combined pill (type · posts · score · date range) updates each time.
    {
        "chapter": "Date: All Time",
        "id": "JMU6",
        "title": "All-Time Sentiment",
        "description": 'Without <code>date_range</code>, the stats pill reflects all-time aggregate values — total post count and average sentiment score across the full dataset. Pair with <code>show_stats</code> and <code>show_place_type</code> to surface these values in the chapter card:<br><pre><code>"show_stats":      True,\n"show_place_type": True</code></pre>',
        "show_stats": True,
        "show_place_type": True,
    },
    {
        "chapter": "Date: Early Period",
        "id": "JMU6",
        "title": "Early Period (2019–2020)",
        "description": '<code>date_range</code> filters the stats calculation to those years only. The pill updates to show the post count and score for just that window, and the active date range is appended so the filter is always visible to the reader:<br><pre><code>"date_range":      (2019, 2020),\n"show_stats":      True,\n"show_place_type": True</code></pre>',
        "date_range": (2019, 2020),
        "show_stats": True,
        "show_place_type": True,
    },
    {
        "chapter": "Date: Recent Period",
        "id": "JMU6",
        "title": "Recent Period (2022–2026)",
        "description": 'Changing <code>date_range</code> on the same <code>id</code> keeps the camera in place while shifting the data window. The combined pill now shows the recalculated score, count, and date window all in one — making the temporal change immediately readable:<br><pre><code>"date_range":      (2022, 2026),\n"show_stats":      True,\n"show_place_type": True</code></pre>',
        "date_range": (2022, 2026),
        "show_stats": True,
        "show_place_type": True,
    },
    {
        "chapter": "Conclusion",
        "id": "JMU6",
        "title": "Conclusion",
        "description": "That covers every available option, but your story does not need all of them. Choose only the fields that serve your narrative, and resist adding features just because they exist. A focused flythrough with 8–10 well-chosen stops communicates more clearly than one that uses every option on every slide.",
    },
    # ── Add more stops by copying and pasting one of the blocks above ─────────────
]

print(str(len(FLYTHROUGH_LOCATIONS)) + " stop(s) defined — run the next cell to validate and export.")


In [ ]:
if "df_work" not in dir() or df_work is None:
    print("⛔ Run the map cell in Section 2 first.")
elif "FLYTHROUGH_LOCATIONS" not in dir():
    print("⛔ Run the cell above to define FLYTHROUGH_LOCATIONS first.")
else:
    # ── Validate every entry and look up coordinates from the map data ────────────
    _id_lookup = df_work.set_index("id")
    _valid_ids = sorted(_id_lookup.index.tolist())

    _VALID_SHOW_DATA = {"all_locations", "jmu_locations", "non_jmu_locations"}

    # ── Build a combined raw DataFrame for date-range filtering ──────────────────
    # df_jmu_raw and df_school_raw are created in Section 1; they have a 'date' column.
    _has_raw = ("df_jmu_raw" in dir() and "df_school_raw" in dir())
    if _has_raw:
        try:
            _raw_combined = pd.concat([df_jmu_raw, df_school_raw], ignore_index=True)
            _raw_combined = _raw_combined.dropna(subset=["place", "roberta_compound", "date"])
            _raw_combined["_year"] = pd.to_datetime(_raw_combined["date"], errors="coerce", format="mixed").dt.year
        except Exception as _e:
            _has_raw = False
            print(f'⚠️  Date filtering unavailable: {_e}')

    chapters = []
    errors   = []

    for i, stop in enumerate(FLYTHROUGH_LOCATIONS):
        chapter = stop.get("chapter", f'Stop {i}')
        loc_id  = stop.get("id")
        if loc_id is None or loc_id not in _id_lookup.index:
            errors.append(f'  [{chapter}] id={loc_id!r} not found — check the hover tooltip for the correct id (e.g. "JMU1", "{_valid_ids[0]}")')
            continue

        show_data = stop.get("show_data", "all_locations")
        if show_data not in _VALID_SHOW_DATA:
            errors.append(f'  [{chapter}] show_data={show_data!r} is not valid — use one of: {sorted(_VALID_SHOW_DATA)}')
            continue

        row = _id_lookup.loc[loc_id]

        # ── Camera center: use custom if provided, otherwise the location's coordinates ──
        center   = stop.get("center") or {}
        cam_lat  = float(center.get("latitude",  row["latitude"]))
        cam_lon  = float(center.get("longitude", row["longitude"]))
        cam_zoom = stop.get("zoom", 15)

        # ── Compute date-filtered sentiment if date_range is specified ────────────
        loc_count = int(row["location_count"])
        loc_score = round(float(row["avg_roberta_compound"]), 3)

        if stop.get("date_range") and _has_raw:
            start_y, end_y = int(stop["date_range"][0]), int(stop["date_range"][1])
            _mask    = (
                (_raw_combined["place"] == row["place"]) &
                (_raw_combined["_year"] >= start_y) &
                (_raw_combined["_year"] <= end_y)
            )
            _filtered = _raw_combined[_mask]
            if len(_filtered) > 0:
                loc_count = len(_filtered)
                loc_score = round(float(_filtered["roberta_compound"].mean()), 3)
            else:
                print(f'  ⚠️  [{chapter}] No data for {row["place"]} in {start_y}–{end_y}; using all-time values.')

        # ── Each stop gets a unique sequential id so Scrollama can distinguish
        # chapters that reference the same location (e.g. the same place in two
        # different date ranges).  Using 'location-{loc_id}' caused duplicate DOM
        # ids, which meant only the first of any repeated location ever fired.
        chapter_entry = {
            "id":          f'chapter-{i}',
            "title":       stop.get("title", row["place"]),
            "description": stop.get("description", ""),
            "image":       f"./{stop["media"]}" if stop.get("media") else None,
            "duration":    2000,
            "transition":  stop.get("transition", "normal"),
            "camera": {
                "latitude":  round(cam_lat,  5),
                "longitude": round(cam_lon,  5),
                "zoom":      cam_zoom,
            },
            "location": {
                "name":          row["place"],
                "latitude":      round(float(row["latitude"]),  5),
                "longitude":     round(float(row["longitude"]), 5),
                "postCount":     loc_count,
                "robertaScore":  loc_score,
                "isJMU":         row["school"] == "JMU",
                "opacity":       stop.get("opacity", 1.0),
                "label":         bool(stop.get("label",           False)),
                "pulse":         bool(stop.get("pulse",           False)),
                "showStats":     bool(stop.get("show_stats",      False)),
                "placeType":     str(row["place_type"]) if pd.notna(row["place_type"]) else None,
                "showPlaceType": bool(stop.get("show_place_type", False)),
            },
            "showData": show_data,
        }

        if stop.get("quote"):
            chapter_entry["quote"] = stop["quote"]

        if stop.get("date_range"):
            chapter_entry["dateRange"] = {
                "start": int(stop["date_range"][0]),
                "end":   int(stop["date_range"][1]),
            }

        if stop.get("caption"):
            chapter_entry["caption"] = stop["caption"]

        if stop.get("hide_place_types"):
            chapter_entry["hidePlaceTypes"] = [str(t) for t in stop["hide_place_types"]]

        chapters.append(chapter_entry)

    if errors:
        print("⛔ Fix these errors before exporting:")
        for e in errors:
            print(e)
    else:
        # ── Wrap chapters in the full flythrough config object ────────────────────
        _ft = FLYTHROUGH_CONFIG if "FLYTHROUGH_CONFIG" in dir() else {}
        _full_config = {
            "theme":            _ft.get("theme",              "light"),
            "footer":           _ft.get("footer",             ""),
            "imageAspectRatio": _ft.get("image_aspect_ratio", None),
            "chapters":         chapters,
        }

        print(f'✅ {len(chapters)} stop(s) validated — run the next cell to write interactive_tour_config.js.\n')
        for stop, ch in zip(FLYTHROUGH_LOCATIONS, chapters):
            chapter    = stop.get("chapter", f'Stop {FLYTHROUGH_LOCATIONS.index(stop)}')
            cam        = ch["camera"]
            opacity    = ch["location"]["opacity"]
            transition = ch["transition"]
            show_data  = ch["showData"]
            score_str  = f'{ch["location"]["robertaScore"]:+.3f}'
            count_str  = str(ch["location"]["postCount"])
            quote_str  = f'  quote="{stop["quote"][:40]}…"' if stop.get("quote") else ""
            label_str  = "  label=True" if stop.get("label") else ""
            date_str   = f'  dates={stop["date_range"][0]}–{stop["date_range"][1]}' if stop.get("date_range") else ""
            print(f'  [{chapter}]  id={stop["id"]}  score={score_str}  posts={count_str}  showData={show_data}  zoom={cam["zoom"]}  opacity={opacity}{date_str}{label_str}  →  {ch["title"]}{quote_str}')


In [ ]:
import os, json

if "chapters" not in dir() or not chapters:
    print("⛔ Run the validation cell above first.")
else:
    # ── Static map settings (students do not change these) ────────────────────────
    _write_config = {
        "tileLayer":   "https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
        "attribution": (
            '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors'
            ' &copy; <a href="https://carto.com/attributions">CARTO</a>'
        ),
        "colorScale": "RdYlGn",
        **_full_config,
    }

    # ── Color classification breaks from the map cell ─────────────────────────────
    # Export the Jenks break points and colour palette so the flythrough template
    # colours bubbles with exactly the same bins as the Plotly map.
    # Color breaks are shared across all pool modes so sentiment colours carry the
    # same meaning whether viewing JMU, partner school, or combined data.
    if "df_work" in dir() and df_work is not None and "_breaks" in dir() and "_palette" in dir():
        _write_config["colorBreaks"]  = [round(float(b), 4) for b in _breaks]
        _write_config["colorPalette"] = list(_palette)
        print(f'✅ colorBreaks: {len(_breaks)} bins exported ({", ".join(f"{b:.3f}" for b in _breaks)})')

    # ── Size classification breaks (combined + per-school) ────────────────────────
    # Combined Jenks breaks power all_locations markers; per-school breaks power
    # jmu_locations and non_jmu_locations so each school's size scale is relative
    # to its own data range rather than the combined dataset.
    if "df_work" in dir() and df_work is not None:
        _k = N_SIZE_CLASSES if "N_SIZE_CLASSES" in dir() else 4
        try:
            import mapclassify as _mc_sz
            if "_jnb_s" in dir():
                _write_config["sizeBreaks"] = [int(b) for b in _jnb_s.bins]
                print(f'✅ sizeBreaks (combined): {_write_config["sizeBreaks"]}')
            _df_jmu_w    = df_work[df_work["school"] == "JMU"]
            _df_nonjmu_w = df_work[df_work["school"] != "JMU"]
            if len(_df_jmu_w) >= _k:
                _jmu_szb = _mc_sz.NaturalBreaks(_df_jmu_w["location_count"].values, k=_k)
                _write_config["jmuSizeBreaks"] = [int(b) for b in _jmu_szb.bins]
                print(f'✅ jmuSizeBreaks: {_write_config["jmuSizeBreaks"]}')
            if len(_df_nonjmu_w) >= _k:
                _nonjmu_szb = _mc_sz.NaturalBreaks(_df_nonjmu_w["location_count"].values, k=_k)
                _write_config["nonJmuSizeBreaks"] = [int(b) for b in _nonjmu_szb.bins]
                print(f'✅ nonJmuSizeBreaks: {_write_config["nonJmuSizeBreaks"]}')
        except Exception as _sze:
            print(f'⚠️  Per-school size breaks skipped: {_sze}')

    # ── All locations from the filtered map (powers the year-range slider) ────────
    # Every bubble that appears on the base map is included so the slider can
    # recolour and resize the full dataset — not just the flythrough chapter stops.
    if "df_work" not in dir() or df_work is None:
        print("⚠️  df_work not found — run the map cell in Section 2 first, then re-run this cell.")
        print("   allLocations will not be exported; the year-range slider will not work.")
    else:
        _write_config["allLocations"] = [
            {
                "name":         r["place"],
                "latitude":     round(float(r["latitude"]), 5),
                "longitude":    round(float(r["longitude"]), 5),
                "postCount":    int(r["location_count"]),
                "robertaScore": round(float(r["avg_roberta_compound"]), 3),
                "isJMU":        bool(r["school"] == "JMU"),
                "school":       str(r["school"]),
                "placeType":    str(r["place_type"]) if pd.notna(r["place_type"]) else None,
            }
            for _, r in df_work.iterrows()
        ]
        print(f'✅ allLocations: {len(_write_config["allLocations"])} locations exported')

    # ── Per-year breakdown for the interactive year-range slider ──────────────────
    # Each unique place gets an array of {year, school, score, count} entries so the
    # template can recompute per-school bubble sizes and colours without a server
    # round-trip. Including 'school' lets the slider correctly show each school's
    # count separately — without it, shared locations (e.g. a city that appears in
    # both JMU and UNC data) would show the combined total in each per-school pool.
    if "df_jmu_raw" in dir() and "df_school_raw" in dir():
        try:
            _raw_yr = pd.concat([df_jmu_raw, df_school_raw], ignore_index=True)
            _raw_yr = _raw_yr.dropna(subset=["place", "roberta_compound", "date"])
            _raw_yr["_year"] = pd.to_datetime(_raw_yr["date"], errors="coerce", format="mixed").dt.year
            _raw_yr = _raw_yr.dropna(subset=["_year"])
            _raw_yr["_year"] = _raw_yr["_year"].astype(int)

            # Group by place + year + school so per-school year-filtered counts stay separate.
            # The template splits these into JMU/non-JMU maps before rebuilding pool markers.
            _yearly_agg = (
                _raw_yr
                .groupby(["place", "_year", "school"])
                .agg(score=("roberta_compound", "mean"), count=("roberta_compound", "size"))
                .reset_index()
            )
            _yearly_dict = {}
            for _place, _grp in _yearly_agg.groupby("place"):
                _yearly_dict[_place] = [
                    {
                        "year":   int(r["_year"]),
                        "score":  round(float(r["score"]), 3),
                        "count":  int(r["count"]),
                        "school": str(r["school"]),
                    }
                    for _, r in _grp.iterrows()
                ]
            _write_config["yearlyData"] = _yearly_dict
            _years = sorted(_raw_yr["_year"].unique())
            print(f'✅ yearlyData: {len(_yearly_dict)} locations × years {_years[0]}–{_years[-1]}')
        except Exception as _ye:
            print(f'⚠️  yearlyData skipped: {_ye}')

    _js_path = "interactive_tour_config.js"
    with open(_js_path, "w", encoding="utf-8") as _f:
        _f.write("// Auto-generated by project_part_3_interactive_tour.ipynb — do not edit by hand\n")
        _f.write("var config = ")
        _f.write(json.dumps(_write_config, indent=2))
        _f.write(";\n")

    print(f'✅ interactive_tour_config.js written ({len(chapters)} chapter(s))')
    print()
    print("To preview: right-click interactive_tour.html in the file browser → Open With → Editor,")
    print("then use the Live Preview button — or open the file directly in your browser.")


---

## 4 Publish & Submit

Follow the checklist below before submitting.

### Final Checklist

- [ ] `interactive_tour_config.js` was written successfully (no ⚠️ warnings above)
- [ ] `interactive_tour.html` loads without errors and animates through all chapters
- [ ] Each chapter has a title, description, image, and correct location ID
- [ ] Images load correctly (or are intentionally absent)
- [ ] The narrative tells a coherent story — a reader with no prior context can follow it
- [ ] `whitepaper.html` is complete (Part 2)
- [ ] `team.html` is complete with all team member names and roles
- [ ] `index.html` project overview is written
- [ ] `publish.sh` has been run and the site is live on GitHub Pages

**You are done. Submit the link to your team's GitHub Pages site.**


---

## Lesson Summary

**📖 1 Follow Along — Load and Aggregate Your Processed Data**
- Loaded your school's cleaned sentiment pickle alongside the JMU baseline and merged them into a single DataFrame
- `_aggregate()` — collapses the per-sentence DataFrame to one row per unique place, computing mean sentiment and total post count
- `pd.concat([df_jmu, df_school], ignore_index=True)` — combines both schools so that Jenks bins are shared across the full dataset

**2 Build Your Map**
- Made seven deliberate design decisions (filter threshold, place-type filter, size classes, color buckets, color scale, bubble sizes, map style) and recorded the reasoning in the Design Brief
- `mapclassify.NaturalBreaks(values, k=n)` — computes Jenks natural-break classification for both size and color
- `px.scatter_map(df, lat=..., lon=..., size=..., color=..., hover_data=...)` — renders the interactive bubble map

**✍️ 3 Plan Your Flythrough Journey**
- Chose 5–10 story stops from the map using hover-tooltip IDs, wrote chapter titles and descriptions, and matched each stop to an uploaded image
- Defined `FLYTHROUGH_LOCATIONS` — the ordered list of chapter dictionaries that drives the interactive tour
- Validated every stop ID and exported `interactive_tour_config.js` with color breaks, all locations, and per-year data for the slider

**4 Publish & Submit**
- Confirmed the config exported successfully, previewed the tour, and ran `publish.sh` to deploy the site to GitHub Pages

➡️ **Next:** [Project Part 4 — Site Styling](project_part_4_global_variables.ipynb)